# Reproduce Paper From Existing Results

**Analysis-only unified notebook. No training, no model downloads, no new experiments.**

This notebook consolidates the final analysis code used across the project and reproduces the paper's reported **numbers, tables, and figures** from the already-generated result archives/checkpoints.

Workflow:

1. locate the canonical result archives already present in the folder;
2. validate ZIP integrity and stored PASS/audit metadata;
3. load the archived raw/summary CSV/JSON outputs;
4. recompute analysis-only summaries where raw outputs are available;
5. regenerate the publication figures/tables;
6. run a final numerical/audit checklist.

### Scientific rule

This notebook does **not** retrain scratch Transformers, rerun Qwen/Pythia extraction, rerun interventions, or rerun HMM simulation. It is the consolidated analysis/reproduction layer over the completed experiments.


**Folder-aware:** result archives may remain inside the existing `Markov_Context_Paper/Part1`, `Part2`, reviewer, or HMM subfolders; they do not need to be copied beside this notebook.


In [ ]:

# ============================================================
# 0. Imports, folder-aware file resolver, output directories
# ============================================================
from pathlib import Path
import io, json, re, zipfile, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path.cwd()
OUTPUT = ROOT / "Reproduced_Paper_Outputs"
TABLE_DIR = OUTPUT / "tables"
FIG_DIR = OUTPUT / "figures"
AUDIT_DIR = OUTPUT / "audit"
for d in [OUTPUT, TABLE_DIR, FIG_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def normalized_name(name):
    p = Path(name)
    stem = re.sub(r"\s+\(\d+\)$", "", p.stem)
    return stem.lower() + p.suffix.lower()

def resolve_file(canonical_name, required=True):
    """
    Resolve a result archive from the notebook folder OR the existing
    Markov_Context_Paper project tree.

    This keeps the user's current directory structure intact; files do not
    need to be copied next to the notebook.
    """
    target = normalized_name(canonical_name)

    search_dirs = [
        ROOT,
        ROOT / "results",
        ROOT / "Markov_Context_Paper",
        ROOT / "Markov_Context_Paper" / "Part1",
        ROOT / "Markov_Context_Paper" / "Part2",
        ROOT / "Markov_Context_Paper" / "Reviewer_Robustness_Checks",
        ROOT / "Markov_Context_Paper" / "Reviewer_A3b_TrueState_Coverage",
        ROOT / "Markov_Context_Paper" / "HMM_Latent_State_Extension",
        ROOT / "Markov_Context_Paper" / "HMM_Oracle_Gap_Fix_GPU",
    ]

    # Also support the common case where the notebook itself is already
    # inside Markov_Context_Paper or one of its subdirectories.
    search_dirs += [
        ROOT / "Part1",
        ROOT / "Part2",
        ROOT / "Reviewer_Robustness_Checks",
        ROOT / "Reviewer_A3b_TrueState_Coverage",
        ROOT / "HMM_Latent_State_Extension",
        ROOT / "HMM_Oracle_Gap_Fix_GPU",
    ]

    # Deduplicate while preserving order.
    seen = set()
    search_dirs = [d for d in search_dirs if not (str(d.resolve()) in seen or seen.add(str(d.resolve())))]

    matches = []
    for d in search_dirs:
        if not d.exists() or not d.is_dir():
            continue
        # Exact filename first.
        exact = d / canonical_name
        if exact.exists():
            return exact
        # Then accept duplicate-download suffixes such as (2), (3), ...
        for p in d.iterdir():
            if p.is_file() and normalized_name(p.name) == target:
                matches.append(p)

    if matches:
        return sorted(matches, key=lambda p: (len(p.parts), p.name))[-1]

    if required:
        searched = "\n  - ".join(str(d) for d in search_dirs)
        raise FileNotFoundError(
            f"Missing required file: {canonical_name}\nSearched:\n  - {searched}"
        )
    return None

def zip_read_csv(zpath, member):
    with zipfile.ZipFile(zpath, "r") as z:
        return pd.read_csv(io.BytesIO(z.read(member)))

def zip_read_json(zpath, member):
    with zipfile.ZipFile(zpath, "r") as z:
        return json.loads(z.read(member).decode("utf-8"))

def zip_find_member(zpath, basename):
    with zipfile.ZipFile(zpath, "r") as z:
        hits = [n for n in z.namelist() if Path(n).name == basename]
    if not hits:
        raise FileNotFoundError(f"{basename} not found inside {zpath.name}")
    if len(hits) > 1:
        # prefer shallowest path
        hits = sorted(hits, key=lambda x: (x.count("/"), x))
    return hits[0]

print("Working folder:", ROOT)
print("Output folder :", OUTPUT)


In [ ]:

# ============================================================
# 1. Resolve canonical completed-experiment archives
# ============================================================
FILES = {
    "markov_final": resolve_file("Markov_Context_FINAL_ANALYSIS.zip"),
    "part2": resolve_file("Part2_Sufficiency_Representation_RESULTS.zip"),
    "causal": resolve_file("Part1_CAUSAL_INTERVENTION_RESULTS.zip"),
    "pythia": resolve_file("Part2_Pythia_CrossFamily_Validation_RESULTS.zip"),
    "true_state": resolve_file("Part2_TrueState_PredictiveReadout_Control_RESULTS.zip"),
    "robustness": resolve_file("Reviewer_Robustness_Checks_RESULTS.zip"),
    "a3b": resolve_file("Reviewer_A3b_TrueState_Coverage_RESULTS.zip"),
    "hmm_gap": resolve_file("HMM_Oracle_Gap_Fix_GPU_RESULTS.zip"),
}

# HMM extension may be a ZIP in the user's folder or the exported CSVs may be loose.
FILES["hmm_extension"] = resolve_file("HMM_Latent_State_Extension_RESULTS.zip", required=False)

for label, p in FILES.items():
    print(f"{label:16s} -> {p.name if p else 'loose CSV fallback'}")


In [ ]:

# ============================================================
# 2. Preflight integrity checks — NO expensive computation
# ============================================================
preflight = []
for label, p in FILES.items():
    if p is None:
        continue
    with zipfile.ZipFile(p, "r") as z:
        bad = z.testzip()
        n = len(z.namelist())
    preflight.append((label, p.name, n, bad is None))
    if bad is not None:
        raise RuntimeError(f"Corrupt ZIP: {p.name}, first bad member: {bad}")

preflight_df = pd.DataFrame(preflight, columns=["component","file","members","zip_integrity"])
display(preflight_df)
assert preflight_df["zip_integrity"].all()

print("\nPRE-FLIGHT: PASS")


## 3. Load the archived paper tables and primary analysis data


In [ ]:

# ============================================================
# 3A. Final Part I / Part II publication tables
# ============================================================
MF = FILES["markov_final"]

main_summary = zip_read_csv(MF, "csv/part1_main_threshold_summary.csv")
threshold_contrasts = zip_read_csv(MF, "csv/part1_threshold_contrasts.csv")
k4_refinement = zip_read_csv(MF, "csv/part1_k4_refinement_summary.csv")
capacity_summary = zip_read_csv(MF, "csv/part1_capacity_summary.csv")
sparse_summary = zip_read_csv(MF, "csv/part1_sparse_summary.csv")
vocab_summary = zip_read_csv(MF, "csv/part1_vocab_summary.csv")
v32_targeted = zip_read_csv(MF, "csv/part1_v32k3_targeted_summary.csv")

p2_final_summary = zip_read_csv(MF, "csv/part2_final_linear_summary.csv")
p2_threshold_contrasts = zip_read_csv(MF, "csv/part2_threshold_contrasts.csv")
p2_model_size = zip_read_csv(MF, "csv/part2_model_size_comparison.csv")
p2_linear_vs_mlp = zip_read_csv(MF, "csv/part2_linear_vs_mlp.csv")
control_summary = zip_read_csv(MF, "csv/part2_input_vs_state_control_summary_final.csv")
cross_part = zip_read_csv(MF, "csv/cross_part_synthesis.csv")

# Raw Part II is retained for the layerwise figure.
P2 = FILES["part2"]
p2 = zip_read_csv(P2, "exports/csv/part2_probe_results_raw.csv")

# Export clean copies into the unified output folder.
tables = {
    "part1_main_threshold_summary.csv": main_summary,
    "part1_threshold_contrasts.csv": threshold_contrasts,
    "part1_k4_refinement_summary.csv": k4_refinement,
    "part1_capacity_summary.csv": capacity_summary,
    "part1_sparse_summary.csv": sparse_summary,
    "part1_vocab_summary.csv": vocab_summary,
    "part1_v32k3_targeted_summary.csv": v32_targeted,
    "part2_final_linear_summary.csv": p2_final_summary,
    "part2_threshold_contrasts.csv": p2_threshold_contrasts,
    "part2_model_size_comparison.csv": p2_model_size,
    "part2_linear_vs_mlp.csv": p2_linear_vs_mlp,
    "part2_input_vs_state_control_summary_final.csv": control_summary,
    "cross_part_synthesis.csv": cross_part,
}
for name, df in tables.items():
    df.to_csv(TABLE_DIR / name, index=False)

print("Loaded and re-exported", len(tables), "paper tables.")
display(main_summary)
display(p2_final_summary.head())


## 4. Reproduce the main publication figures 1–5


In [ ]:

# ============================================================
# 4. REVISED PUBLICATION FIGURES 1–5
# Same analysis variables and plotting logic as the final analysis notebook
# ============================================================

REV_FIG_DIR = FIG_DIR

plt.rcParams.update({
    "font.size": 10.5,
    "axes.titlesize": 11.5,
    "axes.labelsize": 11,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.fontsize": 9,
    "figure.titlesize": 12.5,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.8,
    "lines.markersize": 5.5,
    "errorbar.capsize": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
MODEL_ORDER = sorted(p2_final_summary["model_key"].unique())
MODEL_LABELS = {"qwen05":"Qwen2.5-0.5B","qwen15":"Qwen2.5-1.5B"}
MODEL_COLORS = {mk:_cycle[i % len(_cycle)] for i,mk in enumerate(MODEL_ORDER)}
KS = sorted(set(main_summary["k"].unique()) & set(p2_final_summary["k"].unique()))

def model_label(mk):
    return MODEL_LABELS.get(mk, str(mk))

def style_axis(ax, grid=True):
    if grid:
        ax.grid(True, axis="y", alpha=0.20, linewidth=0.6)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def mark_boundary(ax):
    ax.axvline(0, linestyle=":", linewidth=1.25, alpha=0.75, zorder=0)

def save_figure(fig, stem):
    fig.savefig(REV_FIG_DIR/f"{stem}.png", dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(REV_FIG_DIR/f"{stem}.pdf", bbox_inches="tight", facecolor="white")
    print("Saved:", stem)

# Figure 1
fig,axes=plt.subplots(1,len(KS),figsize=(12.6,3.8),sharey=True)
if len(KS)==1: axes=[axes]
for ax,k in zip(axes,KS):
    d=main_summary[main_summary["k"]==k].sort_values("L").copy()
    ax.plot(d["L"],d["info_gap_mean"],marker="o",linestyle="-",label=r"Information gap $H_L-H^*$",zorder=3)
    if "info_gap_sd" in d.columns:
        lo=np.maximum(0,d["info_gap_mean"]-d["info_gap_sd"].fillna(0))
        hi=d["info_gap_mean"]+d["info_gap_sd"].fillna(0)
        ax.fill_between(d["L"],lo,hi,alpha=0.12,linewidth=0)
    ax.plot(d["L"],d["model_gap_mean"],marker="s",linestyle="--",label="Model / optimization gap",zorder=3)
    if {"model_gap_ci95_low","model_gap_ci95_high"}.issubset(d.columns):
        ax.fill_between(d["L"],d["model_gap_ci95_low"],d["model_gap_ci95_high"],alpha=0.12,linewidth=0)
    ax.axvline(k,linestyle=":",linewidth=1.25,alpha=0.75,zorder=0)
    ax.axhline(0,linewidth=0.8,alpha=0.6,zorder=0)
    ax.set_title(rf"$k={k}$"); ax.set_xlabel(r"Context length $L$"); ax.set_xticks(sorted(d["L"].unique()))
    style_axis(ax)
axes[0].set_ylabel("Gap (nats)")
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(0.5,1.06),ncol=2,frameon=False)
fig.subplots_adjust(top=0.80,bottom=0.18,left=0.08,right=0.99,wspace=0.12)
save_figure(fig,"Fig1_part1_information_decomposition_REVISED"); plt.show()

# Figure 2
fig,axes=plt.subplots(1,len(KS),figsize=(12.6,3.8),sharex=True,sharey=True)
if len(KS)==1: axes=[axes]
for ax,k in zip(axes,KS):
    for mk in MODEL_ORDER:
        d=p2_final_summary[(p2_final_summary["model_key"]==mk)&(p2_final_summary["k"]==k)].sort_values("relative_context")
        ax.errorbar(d["relative_context"],d["state_exact_mean"],yerr=d["state_exact_sd"].fillna(0),
                    marker="o",linestyle="-",color=MODEL_COLORS[mk],capsize=3,label=model_label(mk),zorder=3)
    mark_boundary(ax); ax.set_title(rf"$k={k}$"); ax.set_xlabel(r"Relative context $L-k$")
    ax.set_xticks([-1,0,1]); ax.set_ylim(-0.02,1.05); style_axis(ax)
axes[0].set_ylabel("Exact full-state recovery")
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(0.5,1.06),ncol=len(MODEL_ORDER),frameon=False)
fig.subplots_adjust(top=0.80,bottom=0.18,left=0.08,right=0.99,wspace=0.10)
save_figure(fig,"Fig2_part2_state_recovery_threshold_REVISED"); plt.show()

# Figure 3
fig,axes=plt.subplots(1,len(KS),figsize=(12.6,3.8),sharex=True,sharey=True)
if len(KS)==1: axes=[axes]
for ax,k in zip(axes,KS):
    for mk in MODEL_ORDER:
        d=p2_final_summary[(p2_final_summary["model_key"]==mk)&(p2_final_summary["k"]==k)].sort_values("relative_context")
        ax.errorbar(d["relative_context"],d["next_excess_HL_mean"],yerr=d["next_excess_HL_sd"].fillna(0),
                    marker="o",linestyle="-",color=MODEL_COLORS[mk],capsize=3,label=model_label(mk),zorder=3)
    mark_boundary(ax); ax.axhline(0,linewidth=0.8,alpha=0.6,zorder=0)
    ax.set_title(rf"$k={k}$"); ax.set_xlabel(r"Relative context $L-k$"); ax.set_xticks([-1,0,1]); style_axis(ax)
axes[0].set_ylabel(r"Predictive readout excess CE (nats)")
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(0.5,1.06),ncol=len(MODEL_ORDER),frameon=False)
fig.subplots_adjust(top=0.80,bottom=0.18,left=0.09,right=0.99,wspace=0.10)
save_figure(fig,"Fig3_part2_predictive_readout_gap_REVISED"); plt.show()

# Figure 4
fig,axes=plt.subplots(1,len(KS),figsize=(12.6,3.9),sharex=True,sharey=True)
if len(KS)==1: axes=[axes]
for ax,k in zip(axes,KS):
    for mk in MODEL_ORDER:
        d=control_summary[(control_summary["model_key"]==mk)&(control_summary["k"]==k)].sort_values("relative_context")
        ax.errorbar(d["relative_context"],d["visible_exact_mean"],yerr=d["visible_exact_sd"].fillna(0),
                    marker="o",linestyle="-",color=MODEL_COLORS[mk],capsize=3,zorder=3)
        ax.errorbar(d["relative_context"],d["state_exact_mean"],yerr=d["state_exact_sd"].fillna(0),
                    marker="s",linestyle="--",color=MODEL_COLORS[mk],capsize=3,zorder=3)
    mark_boundary(ax); ax.set_title(rf"$k={k}$"); ax.set_xlabel(r"Relative context $L-k$")
    ax.set_xticks([-1,0,1]); ax.set_ylim(-0.02,1.05); style_axis(ax)
axes[0].set_ylabel("Exact reconstruction accuracy")
model_handles=[Line2D([0],[0],color=MODEL_COLORS[mk],marker="o",linestyle="-",label=model_label(mk)) for mk in MODEL_ORDER]
target_handles=[Line2D([0],[0],color="0.25",marker="o",linestyle="-",label="Visible input"),
                Line2D([0],[0],color="0.25",marker="s",linestyle="--",label="Full state")]
legend1=fig.legend(handles=model_handles,loc="upper center",bbox_to_anchor=(0.37,1.07),ncol=len(model_handles),frameon=False)
fig.add_artist(legend1)
fig.legend(handles=target_handles,loc="upper center",bbox_to_anchor=(0.76,1.07),ncol=2,frameon=False)
fig.subplots_adjust(top=0.78,bottom=0.18,left=0.08,right=0.99,wspace=0.10)
save_figure(fig,"Fig4_part2_visible_vs_state_control_REVISED"); plt.show()

# Figure 5
layer=p2[(p2["probe_type"]=="linear")&(p2["L"]==p2["k"])].copy()
fig,axes=plt.subplots(1,len(KS),figsize=(12.6,3.8),sharex=True,sharey=True)
if len(KS)==1: axes=[axes]
for ax,k in zip(axes,KS):
    for mk in MODEL_ORDER:
        d=layer[(layer["model_key"]==mk)&(layer["k"]==k)].copy()
        a=d.groupby("layer_fraction",as_index=False).agg(y=("state_exact_acc","mean"),e=("state_exact_acc","std"),n=("seed","count")).sort_values("layer_fraction")
        ax.errorbar(a["layer_fraction"],a["y"],yerr=a["e"].fillna(0),marker="o",linestyle="-",
                    color=MODEL_COLORS[mk],capsize=3,label=model_label(mk),zorder=3)
    ax.set_title(rf"$k={k}$"); ax.set_xlabel("Normalized model depth")
    ax.set_xlim(-0.02,1.02); ax.set_ylim(-0.02,1.05); style_axis(ax)
axes[0].set_ylabel("Exact full-state recovery")
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(0.5,1.06),ncol=len(MODEL_ORDER),frameon=False)
fig.subplots_adjust(top=0.80,bottom=0.18,left=0.08,right=0.99,wspace=0.10)
save_figure(fig,"Fig5_part2_layerwise_recovery_REVISED"); plt.show()

print("MAIN FIGURES 1–5: COMPLETE")


## 5. Supplementary true-state and Pythia cross-family figures from archived raw runs


In [ ]:

# ============================================================
# 5. Supplementary figures S1–S2 — analysis only
# ============================================================
def bootstrap_mean_ci(values, B=10000, seed=12345):
    values=np.asarray(values,dtype=float)
    values=values[np.isfinite(values)]
    if len(values)==0: return np.nan,np.nan,np.nan
    mean=values.mean()
    if len(values)==1: return mean,mean,mean
    rng=np.random.default_rng(seed)
    samples=rng.choice(values,size=(B,len(values)),replace=True)
    lo,hi=np.quantile(samples.mean(axis=1),[0.025,0.975])
    return mean,lo,hi

def mean_ci_table(df, group_cols, value_col, B=10000):
    rows=[]
    for keys,d in df.groupby(group_cols):
        if not isinstance(keys,tuple): keys=(keys,)
        mean,lo,hi=bootstrap_mean_ci(d[value_col].values,B=B)
        row=dict(zip(group_cols,keys))
        row.update({"mean":mean,"ci_low":lo,"ci_high":hi,"n":d["seed"].nunique()})
        rows.append(row)
    return pd.DataFrame(rows)

# TRUE-STATE raw runs
true_rows=[]
with zipfile.ZipFile(FILES["true_state"],"r") as z:
    for name in z.namelist():
        if name.startswith("runs/") and name.endswith("/true_state_readout.csv"):
            true_rows.append(pd.read_csv(io.BytesIO(z.read(name))))
true_df=pd.concat(true_rows,ignore_index=True)
true_df["k"]=true_df["k"].astype(int); true_df["seed"]=true_df["seed"].astype(int)

# PYTHIA raw runs
pythia_rows=[]
with zipfile.ZipFile(FILES["pythia"],"r") as z:
    for name in z.namelist():
        if name.startswith("runs/") and name.endswith("/probe_results.csv"):
            pythia_rows.append(pd.read_csv(io.BytesIO(z.read(name))))
pythia_df=pd.concat(pythia_rows,ignore_index=True)
pythia_df["k"]=pythia_df["k"].astype(int); pythia_df["L"]=pythia_df["L"].astype(int); pythia_df["seed"]=pythia_df["seed"].astype(int)
pythia_df["relative_context"]=pythia_df["L"]-pythia_df["k"]

s1_summary=mean_ci_table(true_df,["k","probe_type","training_target"],"excess_over_Hstar_test",B=10000)
s2_summary=mean_ci_table(pythia_df,["k","relative_context","probe_type"],"state_exact_acc",B=10000)
s1_summary.to_csv(TABLE_DIR/"supp_true_state_summary_recomputed.csv",index=False)
s2_summary.to_csv(TABLE_DIR/"supp_pythia_summary_recomputed.csv",index=False)

COLORS=plt.rcParams["axes.prop_cycle"].by_key()["color"]
BOUNDARY_COLOR="0.45"
def _style(ax):
    ax.grid(True,axis="y",alpha=0.20,linewidth=0.6)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

# S1
target_order=["hard_sampled_next_token","oracle_transition_distribution"]
probe_order=["linear","mlp"]
target_labels={"hard_sampled_next_token":"Hard next-token targets",
               "oracle_transition_distribution":"Oracle transition targets"}
probe_labels={"linear":"Linear","mlp":"MLP"}
ks=sorted(true_df["k"].unique())
fig,axes=plt.subplots(1,len(ks),figsize=(12.6,3.9),sharey=True)
if len(ks)==1: axes=[axes]
for ax,k in zip(axes,ks):
    dk=s1_summary[s1_summary["k"]==k]; x=np.arange(len(probe_order))
    for i,target in enumerate(target_order):
        d=dk[dk["training_target"]==target].set_index("probe_type").reindex(probe_order)
        y=d["mean"].values; lo=d["ci_low"].values; hi=d["ci_high"].values
        ax.errorbar(x,y,yerr=np.vstack([y-lo,hi-y]),color=COLORS[i],
                    marker="o" if i==0 else "s",linestyle="-" if i==0 else "--",
                    capsize=3,label=target_labels[target],zorder=3)
    ax.axhline(0,color=BOUNDARY_COLOR,linewidth=0.9,zorder=0)
    ax.set_xticks(x); ax.set_xticklabels([probe_labels[p] for p in probe_order])
    ax.set_title(rf"$k={k}$"); ax.set_xlabel("Predictive readout"); _style(ax)
axes[0].set_ylabel(r"Excess CE over $H^*_{\mathrm{test}}$ (nats)")
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(0.5,1.06),ncol=2,frameon=False)
fig.subplots_adjust(top=0.80,bottom=0.18,left=0.09,right=0.99,wspace=0.12)
fig.savefig(FIG_DIR/"FigS1_true_state_predictive_readout_control.png",dpi=300,bbox_inches="tight")
fig.savefig(FIG_DIR/"FigS1_true_state_predictive_readout_control.pdf",bbox_inches="tight")
plt.show()

# S2
probes=["linear","mlp_final"]; probe_labels={"linear":"Linear probe","mlp_final":"MLP probe"}
ks=sorted(pythia_df["k"].unique())
fig,axes=plt.subplots(1,len(ks),figsize=(12.6,3.9),sharex=True,sharey=True)
if len(ks)==1: axes=[axes]
for ax,k in zip(axes,ks):
    for i,probe in enumerate(probes):
        d=s2_summary[(s2_summary["k"]==k)&(s2_summary["probe_type"]==probe)].sort_values("relative_context")
        y=d["mean"].values; lo=d["ci_low"].values; hi=d["ci_high"].values
        ax.errorbar(d["relative_context"],y,yerr=np.vstack([y-lo,hi-y]),color=COLORS[i],
                    marker="o" if i==0 else "s",linestyle="-" if i==0 else "--",
                    capsize=3,label=probe_labels[probe],zorder=3)
    ax.axvline(0,color=BOUNDARY_COLOR,linestyle=":",linewidth=1.25,zorder=0)
    ax.set_title(rf"$k={k}$"); ax.set_xlabel(r"Relative context $L-k$")
    ax.set_xticks([-1,0,1]); ax.set_ylim(-0.02,1.05); _style(ax)
axes[0].set_ylabel("Exact full-state recovery")
handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc="upper center",bbox_to_anchor=(0.5,1.06),ncol=2,frameon=False)
fig.subplots_adjust(top=0.80,bottom=0.18,left=0.08,right=0.99,wspace=0.10)
fig.savefig(FIG_DIR/"FigS2_pythia_cross_family_validation.png",dpi=300,bbox_inches="tight")
fig.savefig(FIG_DIR/"FigS2_pythia_cross_family_validation.pdf",bbox_inches="tight")
plt.show()

print("SUPPLEMENTARY FIGURES S1–S2: COMPLETE")


## 6. Causal intervention and activation-patching outputs


In [ ]:

# ============================================================
# 6. Causal intervention — recompute plots from archived seed-level results
# ============================================================
CZ=FILES["causal"]
causal_raw=zip_read_csv(CZ,"csv/causal_intervention_seed_level.csv")
causal_summary=zip_read_csv(CZ,"csv/causal_intervention_summary.csv")
patch_summary=zip_read_csv(CZ,"csv/activation_patching_summary.csv")
causal_contrasts=zip_read_csv(CZ,"csv/causal_intervention_paired_contrasts.csv")

for name,df in {
    "causal_intervention_summary.csv":causal_summary,
    "causal_intervention_paired_contrasts.csv":causal_contrasts,
    "activation_patching_summary.csv":patch_summary,
}.items():
    df.to_csv(TABLE_DIR/name,index=False)

KS_CAUSAL=sorted(causal_raw["k"].unique())

# Figure A
rows=[]
for (k,L,intervention),d in causal_raw.groupby(["k","L","intervention"]):
    mean,lo,hi=bootstrap_mean_ci(d["tracking_gain_mean"].values,B=5000,seed=123)
    rows.append({"k":k,"L":L,"intervention":intervention,"mean":mean,"lo":lo,"hi":hi})
plot_df=pd.DataFrame(rows)
fig,ax=plt.subplots(figsize=(8.5,5.5))
x=[]; labels=[]; y=[]; loerr=[]; hierr=[]
for k in KS_CAUSAL:
    for L in [k,k+1]:
        r=plot_df[(plot_df.k==k)&(plot_df.L==L)&(plot_df.intervention=="critical")].iloc[0]
        x.append(len(x)); labels.append(f"k={k}\nL={L}\ncritical"); y.append(r["mean"]); loerr.append(r["mean"]-r["lo"]); hierr.append(r["hi"]-r["mean"])
    r=plot_df[(plot_df.k==k)&(plot_df.L==k+1)&(plot_df.intervention=="irrelevant")].iloc[0]
    x.append(len(x)); labels.append(f"k={k}\nL={k+1}\nirrelevant"); y.append(r["mean"]); loerr.append(r["mean"]-r["lo"]); hierr.append(r["hi"]-r["mean"])
ax.errorbar(x,y,yerr=np.vstack([loerr,hierr]),fmt="o",capsize=4); ax.axhline(0,linewidth=1)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("KL reduction toward target oracle (nats)")
ax.set_title("Native prediction tracks state-changing counterfactuals"); fig.tight_layout()
fig.savefig(FIG_DIR/"FigA_causal_tracking_gain.png",dpi=300); fig.savefig(FIG_DIR/"FigA_causal_tracking_gain.pdf"); plt.show()

# Figure B
rows=[]
for (k,L,intervention),d in causal_raw.groupby(["k","L","intervention"]):
    mean,lo,hi=bootstrap_mean_ci(d["model_shift_js_mean"].values,B=5000,seed=123)
    rows.append({"k":k,"L":L,"intervention":intervention,"mean":mean,"lo":lo,"hi":hi})
shift_df=pd.DataFrame(rows)
fig,ax=plt.subplots(figsize=(8.5,5.5))
x=[]; labels=[]; y=[]; loerr=[]; hierr=[]
for k in KS_CAUSAL:
    L=k+1
    for intervention in ["critical","irrelevant"]:
        r=shift_df[(shift_df.k==k)&(shift_df.L==L)&(shift_df.intervention==intervention)].iloc[0]
        x.append(len(x)); labels.append(f"k={k}\n{intervention}"); y.append(r["mean"]); loerr.append(r["mean"]-r["lo"]); hierr.append(r["hi"]-r["mean"])
ax.errorbar(x,y,yerr=np.vstack([loerr,hierr]),fmt="o",capsize=4); ax.axhline(0,linewidth=1)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("Jensen–Shannon divergence of model predictions (nats)")
ax.set_title("State-changing vs. irrelevant-token intervention"); fig.tight_layout()
fig.savefig(FIG_DIR/"FigB_causal_model_shift_control.png",dpi=300); fig.savefig(FIG_DIR/"FigB_causal_model_shift_control.pdf"); plt.show()

# Figure C
fig,ax=plt.subplots(figsize=(8.5,5.5)); xpos=0; ticks=[]; labels=[]
for k in sorted(patch_summary.k.unique()):
    d=patch_summary[patch_summary.k==k].sort_values("layer")
    for _,r in d.iterrows():
        mean=r["critical_minus_irrelevant_patch_gain_mean_mean_over_seeds"]
        lo=r["critical_minus_irrelevant_patch_gain_mean_ci95_lo"]; hi=r["critical_minus_irrelevant_patch_gain_mean_ci95_hi"]
        ax.errorbar([xpos],[mean],yerr=[[mean-lo],[hi-mean]],fmt="o",capsize=4)
        ticks.append(xpos); labels.append(f"k={k}\nlayer {int(r['layer'])}"); xpos+=1
ax.axhline(0,linewidth=1); ax.set_xticks(ticks); ax.set_xticklabels(labels)
ax.set_ylabel("Critical − irrelevant activation-patch gain (nats)")
ax.set_title("Layer-wise causal steering of native prediction"); fig.tight_layout()
fig.savefig(FIG_DIR/"FigC_activation_patching.png",dpi=300); fig.savefig(FIG_DIR/"FigC_activation_patching.pdf"); plt.show()

display(causal_summary)
display(patch_summary)
print("CAUSAL OUTPUTS: COMPLETE")


## 7. Reviewer robustness A1–A4 and A3b


In [ ]:

# ============================================================
# 7. Reviewer robustness tables and figures from completed results
# ============================================================
RZ=FILES["robustness"]
prefix="Reviewer_Robustness_Checks/"
a1=zip_read_csv(RZ,prefix+"csv/A1_source_supported_counterfactual_summary.csv")
a2=zip_read_csv(RZ,prefix+"csv/A2_token_map_robustness_summary.csv")
a3=zip_read_csv(RZ,prefix+"csv/A3_probe_capacity_data_sweep_contrasts.csv")
a4=zip_read_csv(RZ,prefix+"csv/A4_boundary_gap_summary.csv")
for name,df in {"A1_source_supported_counterfactual_summary.csv":a1,
                "A2_token_map_robustness_summary.csv":a2,
                "A3_probe_capacity_data_sweep_contrasts.csv":a3,
                "A4_boundary_gap_summary.csv":a4}.items():
    df.to_csv(TABLE_DIR/name,index=False)

# A4 plot
fig,ax=plt.subplots(figsize=(7.4,4.6))
ax.errorbar(a4["k"],a4["gap_mean"],yerr=a4["gap_sd"],fmt="o-",capsize=4,label=r"$H_{k-1}-H^*$")
ax.axhline(0,linewidth=1); ax.set_xticks(a4["k"]); ax.set_xlabel("Markov order k"); ax.set_ylabel("Information gap (nats)")
ax.set_title("Strict positive boundary gap across sampled kernels"); ax.legend(); fig.tight_layout()
fig.savefig(FIG_DIR/"A4_boundary_gap_reproduced.png",dpi=300); plt.show()

A3Z=FILES["a3b"]
p="Reviewer_A3b_TrueState_Coverage/"
a3b_cov=zip_read_csv(A3Z,p+"csv/A3b_coverage_summary.csv")
a3b_raw=zip_read_csv(A3Z,p+"csv/A3b_true_state_coverage_raw.csv")
a3b_summary=zip_read_csv(A3Z,p+"csv/A3b_true_state_coverage_summary.csv")
a3b_cov.to_csv(TABLE_DIR/"A3b_coverage_summary.csv",index=False)
a3b_summary.to_csv(TABLE_DIR/"A3b_true_state_coverage_summary.csv",index=False)

# A3b stationary-mass coverage
fig,ax=plt.subplots(figsize=(7.4,4.6))
for k,d in a3b_cov.groupby("k"):
    ax.errorbar(d["n_train"],d["stationary_mass_coverage_mean"],
                yerr=d["stationary_mass_coverage_sd"],marker="o",label=f"k={k}",capsize=3)
ax.set_xscale("log",base=2); ax.set_xlabel("Training examples"); ax.set_ylabel("Stationary-mass coverage")
ax.set_title("True-state coverage vs. data size"); ax.legend(); fig.tight_layout()
fig.savefig(FIG_DIR/"A3b_stationary_mass_coverage_reproduced.png",dpi=300); plt.show()

display(a1)
display(a4)
display(a3b_cov)
print("REVIEWER ROBUSTNESS OUTPUTS: COMPLETE")


## 8. HMM extension and corrected same-history information gap


In [ ]:

# ============================================================
# 8. HMM — use completed extension summaries + corrected KL gap
# ============================================================
def load_hmm_loose_or_zip(basename):
    # Prefer loose curated exports resolved from the repository, including results/.
    p=resolve_file(basename, required=False)
    if p is not None:
        return pd.read_csv(p)
    hz=FILES.get("hmm_extension")
    if hz is not None:
        member=zip_find_member(hz,basename)
        return zip_read_csv(hz,member)
    raise FileNotFoundError(
        f"{basename} not found loose and HMM_Latent_State_Extension_RESULTS.zip is absent."
    )

native_summary=load_hmm_loose_or_zip("HMM_native_oracle_summary.csv")
probe_summary=load_hmm_loose_or_zip("HMM_probe_summary.csv")

HG=FILES["hmm_gap"]
gap_summary=zip_read_csv(HG,"HMM_Oracle_Gap_Fix_GPU/csv/HMM_information_gap_same_sequence_GPU_summary.csv")
gap_raw=zip_read_csv(HG,"HMM_Oracle_Gap_Fix_GPU/csv/HMM_information_gap_same_sequence_GPU.csv")

native_summary.to_csv(TABLE_DIR/"HMM_native_oracle_summary.csv",index=False)
probe_summary.to_csv(TABLE_DIR/"HMM_probe_summary.csv",index=False)
gap_summary.to_csv(TABLE_DIR/"HMM_information_gap_same_sequence_GPU_summary.csv",index=False)

# HMM predictive CE
fig,ax=plt.subplots(figsize=(7.5,4.8))
ax.plot(native_summary.L,native_summary.oracle_ce_mean,marker="o",label="Exact finite-context oracle")
ax.plot(native_summary.L,native_summary.model_ce_mean,marker="o",label="Scratch Transformer")
ax.axhline(native_summary.long_oracle_mean.mean(),linestyle="--",label="Long-history oracle")
ax.set_xscale("log",base=2); ax.set_xlabel("Context length L"); ax.set_ylabel("Cross-entropy (nats)")
ax.set_title("HMM predictive performance vs exact Bayesian oracle"); ax.legend(); fig.tight_layout()
fig.savefig(FIG_DIR/"HMM_oracle_vs_model_ce.png",dpi=300); plt.show()

# HMM representation/readout
fig,ax=plt.subplots(figsize=(7.5,4.8))
ax.plot(probe_summary.L,probe_summary.posterior_KL_mean,marker="o",label="Latent posterior probe KL")
ax.plot(probe_summary.L,probe_summary.readout_excess_mean,marker="o",label="Predictive readout excess CE")
ax.set_xscale("log",base=2); ax.set_xlabel("Context length L"); ax.set_ylabel("Gap (nats)")
ax.set_title("Latent-state accessibility and predictive readout"); ax.legend(); fig.tight_layout()
fig.savefig(FIG_DIR/"HMM_representation_readout.png",dpi=300); plt.show()

# Corrected information-gap decomposition: same-history KL + model excess over finite-context oracle
hmm_merge=native_summary[["L","model_excess_mean"]].merge(gap_summary[["L","information_gap_mean"]],on="L",how="inner")
fig,ax=plt.subplots(figsize=(7.5,4.8))
ax.plot(hmm_merge.L,hmm_merge.information_gap_mean,marker="o",label="Finite-context information gap (same-history KL)")
ax.plot(hmm_merge.L,hmm_merge.model_excess_mean,marker="o",label=r"Model excess over $H_L$")
ax.set_xscale("log",base=2); ax.set_xlabel("Context length L"); ax.set_ylabel("Gap (nats)")
ax.set_title("Information limitation vs model/optimization gap"); ax.legend(); fig.tight_layout()
fig.savefig(FIG_DIR/"HMM_gap_decomposition_CORRECTED.png",dpi=300)
fig.savefig(FIG_DIR/"HMM_gap_decomposition_CORRECTED.pdf")
plt.show()

display(gap_summary)
print("HMM OUTPUTS: COMPLETE")


## 9. Final audit: verify the archived experiments and key numerical fingerprints


In [ ]:

# ============================================================
# 9. Final PASS / numerical fingerprint audit
# ============================================================
checks=[]

def add(name, ok, detail=""):
    checks.append({"check":name,"status":"PASS" if bool(ok) else "FAIL","detail":str(detail)})

# Stored experiment audits
causal_audit=zip_read_json(FILES["causal"],"causal_intervention_audit.json")
rob_audit=zip_read_json(FILES["robustness"],"Reviewer_Robustness_Checks/reviewer_robustness_audit.json")
a3b_audit=zip_read_json(FILES["a3b"],"Reviewer_A3b_TrueState_Coverage/A3b_audit.json")
hmm_gap_audit=zip_read_json(FILES["hmm_gap"],"HMM_Oracle_Gap_Fix_GPU/HMM_oracle_gap_fix_GPU_audit.json")
p2_audit=zip_read_json(FILES["part2"],"exports/part2_integrity_audit.json")
pythia_audit=zip_read_json(FILES["pythia"],"exports/pythia_crossfamily_integrity_audit.json")
true_audit=zip_read_json(FILES["true_state"],"exports/true_state_integrity_audit.json")

add("Causal audit",causal_audit.get("status")=="PASS",causal_audit.get("status"))
add("Reviewer robustness audit",rob_audit.get("status")=="PASS",rob_audit.get("status"))
add("A3b audit",a3b_audit.get("status")=="PASS",a3b_audit.get("status"))
add("Corrected HMM-gap audit",hmm_gap_audit.get("status")=="PASS" and hmm_gap_audit.get("all_KL_nonnegative_with_tolerance") is True,hmm_gap_audit.get("status"))
add("Part II integrity",p2_audit.get("done_configs")==p2_audit.get("expected_configs") and p2_audit.get("nonfinite_rows")==0,p2_audit)
add("Pythia integrity",pythia_audit.get("done_configs")==pythia_audit.get("expected_configs") and pythia_audit.get("all_summary_cells_n10") is True,pythia_audit)
add("True-state integrity",true_audit.get("done_runs")==true_audit.get("expected_runs") and true_audit.get("nonfinite_rows")==0,true_audit)

# Key paper numerical fingerprints
a4_lookup=a4.set_index("k")
add("A4 k=2 boundary gap",np.isclose(a4_lookup.loc[2,"gap_mean"],0.300477,atol=5e-6),a4_lookup.loc[2,"gap_mean"])
add("A4 k=3 boundary gap",np.isclose(a4_lookup.loc[3,"gap_mean"],0.303743,atol=5e-6),a4_lookup.loc[3,"gap_mean"])
add("A4 k=4 boundary gap",np.isclose(a4_lookup.loc[4,"gap_mean"],0.300868,atol=5e-6),a4_lookup.loc[4,"gap_mean"])

a1_lookup=a1.set_index("k")
add("A1 k=2 supported gain",np.isclose(a1_lookup.loc[2,"supported_gain_mean"],0.771175,atol=5e-6),a1_lookup.loc[2,"supported_gain_mean"])
add("A1 k=3 supported gain",np.isclose(a1_lookup.loc[3,"supported_gain_mean"],0.751429,atol=5e-6),a1_lookup.loc[3,"supported_gain_mean"])
add("A1 k=4 supported gain",np.isclose(a1_lookup.loc[4,"supported_gain_mean"],0.721981,atol=5e-6),a1_lookup.loc[4,"supported_gain_mean"])

gap_lookup=gap_summary.set_index("L")
for L,ref in {2:6.646039e-4,4:5.411595e-5,8:4.774972e-7,16:5.761106e-11}.items():
    add(f"HMM corrected gap L={L}",np.isclose(gap_lookup.loc[L,"information_gap_mean"],ref,rtol=2e-6,atol=1e-15),gap_lookup.loc[L,"information_gap_mean"])

# Output existence
expected_figs=[
    "Fig1_part1_information_decomposition_REVISED.png",
    "Fig2_part2_state_recovery_threshold_REVISED.png",
    "Fig3_part2_predictive_readout_gap_REVISED.png",
    "Fig4_part2_visible_vs_state_control_REVISED.png",
    "Fig5_part2_layerwise_recovery_REVISED.png",
    "FigS1_true_state_predictive_readout_control.png",
    "FigS2_pythia_cross_family_validation.png",
    "FigA_causal_tracking_gain.png",
    "FigB_causal_model_shift_control.png",
    "FigC_activation_patching.png",
    "HMM_oracle_vs_model_ce.png",
    "HMM_representation_readout.png",
    "HMM_gap_decomposition_CORRECTED.png",
]
for f in expected_figs:
    add("Figure exists: "+f,(FIG_DIR/f).exists(),f)

audit_df=pd.DataFrame(checks)
display(audit_df)
overall=(audit_df["status"]=="PASS").all()

audit_payload={"status":"PASS" if overall else "FAIL","checks":checks}
(AUDIT_DIR/"reproduction_audit.json").write_text(json.dumps(audit_payload,indent=2),encoding="utf-8")
audit_df.to_csv(AUDIT_DIR/"reproduction_audit.csv",index=False)

print("\n"+"="*72)
print("PAPER REPRODUCTION FROM EXISTING RESULTS:", "PASS" if overall else "FAIL")
print("="*72)
print("Figures:", FIG_DIR)
print("Tables :", TABLE_DIR)
print("Audit  :", AUDIT_DIR)


## Result

A successful run should end with:

```text
PAPER REPRODUCTION FROM EXISTING RESULTS: PASS
```

This confirms that the completed experiment archives were readable, their stored integrity audits were consistent, the analysis tables/figures were regenerated, and the key numerical fingerprints used in the revised manuscript match the archived results.

**No training or new scientific experiment is performed by this notebook.**
